# 01 — EDA Oficial (sanity check)

**Objetivo:** validar o input antes de modelar. Não é EDA exaustiva — só o suficiente para garantir que `load_features()` produz dados coerentes, sem surpresas em outliers, missing ou na distribuição do target engenheirado.

**Input:** `data/processed/vin_features.csv` (175k VINs × 22 colunas).
**Output esperado:** confirmação de que (1) o target `churned` tem taxa ~30-40%, (2) o `reference_date` é fixo (não `datetime.now()`), (3) `km_max` está corretamente clipado a 500k, (4) o split entre VINs treináveis vs holdout é consistente.

> Próximo notebook: [04_classification.ipynb](04_classification.ipynb).

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.features import load_features, split_trainable, CLASSIFIER_FEATURES

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

bundle = load_features()
df = bundle.df
print(f"Shape: {df.shape}")
print(f"Reference date (max ServiceDate): {bundle.reference_date.date()}")
print(f"Total VINs: {bundle.n_total:,}")
print(f"Trainable VINs (tenure >= 365 days): {bundle.n_trainable:,}")
print(f"Trainable share: {bundle.n_trainable / bundle.n_total:.1%}")

## 1. Distribuição do target engenheirado

**Definição:** `churned = days_since_last_service > 365`, onde `days_since_last_service = reference_date - last_service_date`. `reference_date` é o `max(ServiceDate)` do dataset (não `datetime.now()`) — escolha feita para reprodutibilidade.

Esperamos taxa global de churn entre 30–40% no universo treinável. Por modelo, KA tende a churnar mais (ticket menor → mais defecção para oficinas independentes); RANGER tende a manter retenção mais alta.

In [ ]:
df_train, df_holdout = split_trainable(df)

print("--- Taxa global de churn ---")
print(df["churned"].value_counts(normalize=True).rename({0: "retained", 1: "churned"}))
print()
print("--- Taxa de churn apenas em VINs treináveis (tenure >= 365d) ---")
print(df_train["churned"].value_counts(normalize=True).rename({0: "retained", 1: "churned"}))
print()
print("--- Top 8 modelos: taxa de churn por model_name (em treináveis) ---")
churn_by_model = (
    df_train.groupby("model_name")
    .agg(n=("churned", "size"), churn_rate=("churned", "mean"))
    .sort_values("n", ascending=False)
    .head(8)
)
print(churn_by_model.round(3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

churn_by_model.sort_values("churn_rate")["churn_rate"].plot.barh(
    ax=axes[0], color="#c44e52"
)
axes[0].set_title("Taxa de churn por modelo (top 8 por volume)")
axes[0].set_xlabel("Churn rate")
axes[0].axvline(df_train["churned"].mean(), color="black", linestyle="--", label="média global")
axes[0].legend()

churn_by_year = (
    df_train.groupby("model_year")
    .agg(n=("churned", "size"), churn_rate=("churned", "mean"))
)
axes[1].plot(churn_by_year.index, churn_by_year["churn_rate"], marker="o")
axes[1].set_title("Taxa de churn por model_year (treináveis)")
axes[1].set_xlabel("Model year")
axes[1].set_ylabel("Churn rate")
plt.tight_layout()
plt.show()

## 2. Curva da Morte (proxy via `events_count`)

O HANDOFF documenta a Curva da Morte direto na coluna `MaintenanceNumber` do dataset bruto. Como o `vin_features.csv` já está agregado por VIN, usamos `events_count` como proxy: quantos VINs sobrevivem até ter 2, 3, 4, 5+ eventos. Se a queda for monotônica e parecida com o número do HANDOFF (1ª 31% → 2ª 22% → 3ª 15% → 4ª 9% → 5ª 6%), está coerente.

In [ ]:
events_dist = df["events_count"].value_counts(normalize=True).sort_index()
death_curve = pd.DataFrame({
    "events_count": events_dist.index,
    "share_of_vins": events_dist.values,
})
death_curve = death_curve[death_curve["events_count"] <= 8]
print(death_curve.assign(share_of_vins=lambda x: (x["share_of_vins"] * 100).round(2).astype(str) + "%"))

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(death_curve["events_count"], death_curve["share_of_vins"] * 100, color="#4c72b0")
ax.set_xlabel("Número de eventos por VIN (proxy de MaintenanceNumber)")
ax.set_ylabel("% de VINs (no dataset completo)")
ax.set_title("Curva da Morte — distribuição de eventos por VIN")
for x, y in zip(death_curve["events_count"], death_curve["share_of_vins"] * 100):
    ax.text(x, y + 0.4, f"{y:.1f}%", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

## 3. Datas e `days_since_last_service`

Validar que `days_since_last_service` é coerente com a definição do target. A linha vertical de 365 dias separa retidos (esquerda) de churnados (direita).

In [ ]:
print("--- days_since_last_service (universo treinável) ---")
print(df_train["days_since_last_service"].describe().round(1))

fig, ax = plt.subplots(figsize=(12, 4))
ax.hist(df_train["days_since_last_service"], bins=80, color="#4c72b0", edgecolor="white")
ax.axvline(365, color="red", linestyle="--", linewidth=2, label="threshold churn (365d)")
ax.set_xlabel("Days since last service")
ax.set_ylabel("VINs")
ax.set_title("Distribuição de days_since_last_service (treináveis)")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Outliers e missing após `load_features()`

Após a passagem pelo pipeline em [src/features.py](../src/features.py), `km_max` deve estar clipado a 500.000 e os NaN em colunas de gap devem estar imputados.

In [ ]:
print(f"--- km_max stats (após clip) ---")
print(df["km_max"].describe().round(0))
assert df["km_max"].max() <= 500_000, "km_max ainda contém outliers acima de 500k!"

print()
print("--- Missing por feature do classificador ---")
missing_counts = df[CLASSIFIER_FEATURES].isna().sum()
missing = missing_counts[missing_counts > 0]
if len(missing) == 0:
    print("OK: nenhuma feature do classificador tem NaN após imputação.")
else:
    print(missing)

## 5. Trainable vs holdout

VINs com `tenure_days < 365` são excluídos do treino (não houve tempo suficiente para observar churn). Vamos comparar as duas populações.

In [ ]:
comp = pd.DataFrame({
    "trainable": df_train[["events_count", "tenure_days", "km_max", "is_single_event"]].mean(),
    "holdout":   df_holdout[["events_count", "tenure_days", "km_max", "is_single_event"]].mean(),
}).round(2)
comp["delta_pct"] = ((comp["trainable"] - comp["holdout"]) / comp["holdout"].replace(0, np.nan) * 100).round(1)
print("--- Médias por grupo ---")
print(comp)
print()
print(f"Trainable: {len(df_train):,} VINs ({len(df_train)/len(df):.1%})")
print(f"Holdout (tenure < 365): {len(df_holdout):,} VINs ({len(df_holdout)/len(df):.1%})")

## Conclusão

Se todos os checks acima passarem:
- ✅ Target tem distribuição razoável (não está degenerado em 0% ou 100%)
- ✅ `reference_date` é fixo (não muda entre execuções)
- ✅ `km_max` clipado a 500k
- ✅ Sem NaN nas features do classificador
- ✅ Trainable é maioria do dataset (~80%)

→ Seguir para [04_classification.ipynb](04_classification.ipynb).